# Build a diffusion model: MNIST workshop

Work through the numbered TODOs in this notebook and the two exercise modules.
Imports, data, plotting, cosine beta construction, attention, and model dimensions
are provided. The TODOs focus on the tensor operations that make diffusion work.

1. **Noise and signal:** TODOs 1–3 in `diffusion_exercises.py`.
2. **Timestep-aware U-Net:** TODOs 4–6 in `unet_exercises.py`.
3. **Learning and generation:** TODO 7 here, TODOs 8–9 in `diffusion_exercises.py`,
   and TODO 10 here.

Each unfinished exercise raises a numbered `NotImplementedError`. Complete the
exercise named in the message, **restart the kernel**, and rerun the notebook.
The plots are your visual feedback; there are no assertion exercises.

Keep `RUN_TRAINING = False` to use the shared pretrained weights while implementing
the math and sampler. After completing TODO 7, optionally enable a short training
run. You do not need to wait for a full model to train during the meeting.
The completed reference is in `../mnist-implementation`; try the exercises first.

## 1. Setup and real digits

Keep the `mnist-workshop` and `mnist-implementation` folders beside each other.
The workshop shares the demo's checkpoint and cached data, but imports only the
exercise modules. Run from this folder or the repository root.

After editing a Python module, restart the kernel so your changes are imported.
The notebook intentionally stops at unfinished TODOs. See `README.md` for the
suggested meeting plan.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm

# Support launching from either the repository root or this notebook's folder.
TUTORIAL_DIR = Path.cwd()
if (TUTORIAL_DIR / "mnist-workshop").is_dir():
    TUTORIAL_DIR = TUTORIAL_DIR / "mnist-workshop"
sys.path.insert(0, str(TUTORIAL_DIR))

from diffusion_exercises import Diffusion
from unet_exercises import UNet, timestep_embedding

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# Start with pretrained weights; training is optional after TODO 7.
RUN_TRAINING = False
TRAIN_EPOCHS = 1
TRAIN_SUBSET = 2048  # Set None and use 25 epochs for a longer experiment.
NUM_SAMPLES = 16

ASSET_DIR = TUTORIAL_DIR.parent / "mnist-implementation"

checkpoint = torch.load(
    ASSET_DIR / "checkpoints" / "mnist_demo.pt",
    map_location="cpu", weights_only=True,
)
config = checkpoint["config"]
torch.manual_seed(config["seed"])
print(f"PyTorch {torch.__version__} | device: {device}")
print(checkpoint["training_notes"])


def show_images(images, title, nrow=8, figsize=(9, 4)):
    """Display [-1, 1] images; clip only the display, never the noising process."""
    images = (images.detach().cpu().clamp(-1, 1) + 1) / 2
    grid = make_grid(images, nrow=nrow, padding=2, pad_value=1)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(grid.permute(1, 2, 0).numpy(), vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
    plt.show()

MNIST images have shape `(1, 28, 28)`. Map their pixels from $[0,1]$ to $[-1,1]$.
This centers their range; it does not make the digits Gaussian. We do not use digit
labels because the model generates digits without being asked for a particular class.
The checkpoint includes eight real examples so the demo works offline.

In [ ]:
example_images = checkpoint["example_images"]
show_images(example_images, "Real MNIST digits", figsize=(10, 2))

# The dataset and loader are only needed when training is explicitly enabled.
if RUN_TRAINING:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])
    train_set = datasets.MNIST(
        root=ASSET_DIR / "data", train=True, download=True, transform=transform
    )
    if TRAIN_SUBSET is not None:
        generator = torch.Generator().manual_seed(config["seed"])
        indices = torch.randperm(len(train_set), generator=generator)[:TRAIN_SUBSET]
        train_set = Subset(train_set, indices.tolist())
    train_loader = DataLoader(
        train_set, batch_size=config["batch_size"], shuffle=True, num_workers=0
    )

## 2. How noise replaces an image

At each forward step, retain a little less signal and add Gaussian noise:

$$q(x_t\mid x_{t-1}) = \mathcal N(\sqrt{\alpha_t}x_{t-1},\beta_t I),
\qquad \alpha_t=1-\beta_t,\qquad \bar\alpha_t=\prod_{s=0}^{t}\alpha_s.$$

We use a **cosine** schedule: it keeps more signal through the middle of the
process than a linear schedule. This gives training more examples
where a recognizable digit and substantial noise are both present.

Rather than choosing each $\beta_t$ directly, define a smooth signal-retention curve
at the step boundaries, then take ratios of adjacent values:

$$f(s)=\cos^2\!\left(\frac{s/T+0.008}{1.008}\frac{\pi}{2}\right),
\qquad \beta_t=\min\!\left(1-\frac{f(t+1)}{f(t)},\,0.999\right).$$

The upper limit avoids a step that destroys all signal exactly. $\beta_t$ is the
variance added in one step; $\alpha_t$ is the one-step signal factor;
$\bar\alpha_t$ describes accumulated signal retention. The three plots below show
the actual coefficients used for training and generation.

**Index convention:** code indices run from `0` to `T-1`. Index `0` is the
first *slightly noisy* image, not the clean image. We call the original image
$x_{\mathrm{clean}}$ and set $\bar\alpha_{-1}=1$.

**Exercise:** Complete TODO 1 before constructing the schedule below.

In [ ]:
diffusion = Diffusion(timesteps=config["timesteps"], device=device)

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, values, title in zip(
    axes,
    [diffusion.betas, diffusion.alphas, diffusion.alpha_bar],
    [r"$\beta_t$: added noise variance", r"$\alpha_t$: one-step retention",
     r"$\bar\alpha_t$: cumulative retention"],
):
    ax.plot(values.cpu().numpy())
    ax.set(title=title, xlabel="Timestep index")
    ax.grid(alpha=0.3)
# A log axis makes both the small early betas and the large final beta visible.
axes[0].set_yscale("log")
plt.tight_layout()
plt.show()

### Jump directly to any noise level

We do not need to loop through the forward steps. Their combined distribution is:

$$x_t=\sqrt{\bar\alpha_t}\,x_{\mathrm{clean}}
      +\sqrt{1-\bar\alpha_t}\,\varepsilon,
\qquad\varepsilon\sim\mathcal N(0,I).$$

`Diffusion.add_noise` implements this equation. `extract` selects one coefficient
per image and reshapes `(B,)` to `(B, 1, 1, 1)` for multiplication with image tensors.
During training, each image can therefore use a different random timestep.

Below, rows are different digits and columns are noise levels. We reuse the same
noise for each digit across columns to isolate the effect of the coefficients.
These are samples from the marginal distributions, **not** a simulated Markov trajectory.

**Exercise:** Complete TODOs 2 and 3 before plotting the noisy digits.

In [ ]:
clean = example_images[:4].to(device)
noise = torch.randn_like(clean)
forward_steps = [int(fraction * (diffusion.timesteps - 1))
                 for fraction in [0, 0.05, 0.1, 0.2, 0.4, 0.7, 1]]
fig, axes = plt.subplots(len(clean), len(forward_steps) + 1, figsize=(12, 6))
for row in range(len(clean)):
    axes[row, 0].imshow(clean[row, 0].cpu(), cmap="gray", vmin=-1, vmax=1)
    axes[row, 0].set_title("Clean" if row == 0 else "")
for column, step in enumerate(forward_steps, start=1):
    t = torch.full((len(clean),), step, device=device, dtype=torch.long)
    noisy = diffusion.add_noise(clean, t, noise).cpu()
    for row in range(len(clean)):
        axes[row, column].imshow(noisy[row, 0], cmap="gray", vmin=-1, vmax=1)
        axes[row, column].set_title(f"t = {step}" if row == 0 else "")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Signal fades as noise takes over (display clipped to [-1, 1])")
plt.tight_layout()
plt.show()

**Discuss:** why does $\alpha_t$ stay close to one while $\bar\alpha_t$ approaches zero?
Use the plots to locate the region where digits become hard to distinguish.
Keep the default 1,000 steps while learning. Reducing this count does not speed up
training batches, since each image is noised directly at one random timestep.
Training and sampling use the same cosine schedule.

## 3. The noise predictor

The U-Net predicts $\varepsilon_\theta(x_t,t)$, a noise image of the same shape as $x_t$.
The architecture in `unet_exercises.py` is deliberately explicit:

| Stage | Spatial size | Role |
|---|---|---|
| Encoder | 28 → 14 → 7 | Build broader image features |
| Bottleneck | 7 × 7 | Share global information with one attention block |
| Decoder | 7 → 14 → 28 | Combine features with encoder skips to recover detail |

Each encoder and decoder resolution has **two residual blocks**. The second
block adds capacity to combine local strokes into coherent shapes without adding
a new type of layer. Residual branches start at zero, so they initially preserve
the skip connection.

Every residual block receives a timestep embedding. Sine and cosine features at
multiple frequencies encode the integer timestep; an MLP turns them into learned
features, and each block adds a projected per-channel shift. The heatmap shows the
fixed embedding **before** the learned MLP. Sine features occupy the first half,
cosine features the second half.

**Exercise:** Complete TODO 4 for the embedding plot, then TODOs 5 and 6 for the model forward pass.

In [ ]:
model = UNet(base_channels=config["base_channels"]).to(device)
if not RUN_TRAINING:
    model.load_state_dict(checkpoint["model"])
    model.eval()

embeddings = timestep_embedding(
    torch.arange(config["timesteps"]), config["base_channels"]
)
fig, ax = plt.subplots(figsize=(10, 3))
image = ax.imshow(embeddings.T.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set(xlabel="Timestep index", ylabel="Embedding feature", title="Timestep embeddings")
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.show()
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Learn to predict the added noise

For each batch, choose one random timestep per image, draw noise, and create noisy
training inputs. The target is the noise we just drew:

$$\mathcal L=\mathbb E_{x_{\mathrm{clean}},t,\varepsilon}
\left[\|\varepsilon-\varepsilon_\theta(x_t,t)\|^2\right].$$

Complete TODO 7 inside the loop below. It is skipped until training is enabled. To try training,
set `RUN_TRAINING = True` in setup and restart/run all cells. This starts a new model.
The short default training run demonstrates optimization but will produce rough
samples. Use all 60,000 examples and more epochs for quality.

Training writes a separate `checkpoints/training.pt`; the demo checkpoint is never
changed by this notebook. Loss values from different schedules are not directly
comparable, so inspect images as well.

In [ ]:
if RUN_TRAINING:
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    loss_history, epoch_losses = [], []
    for epoch in range(TRAIN_EPOCHS):
        model.train()
        total_loss, images_seen = 0.0, 0
        progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}")
        for clean, _ in progress:
            clean = clean.to(device)
            # TODO 7: Build one noise-prediction training step.
            # 1. Sample one integer timestep per image with torch.randint.
            # 2. Draw Gaussian noise, then call diffusion.add_noise.
            # 3. Predict noise with model(noisy, t) and compute F.mse_loss against the target noise.
            # 4. Zero gradients, backpropagate, clip gradient norm to 1.0, and step the optimizer.
            # Assign the scalar loss tensor to loss; the logging code below uses it.
            raise NotImplementedError("TODO 7: Build one noise-prediction training step.")

            loss_value = loss.item()
            loss_history.append(loss_value)
            total_loss += loss_value * len(clean)
            images_seen += len(clean)
            progress.set_postfix(mse=f"{total_loss / images_seen:.4f}")
        epoch_losses.append(total_loss / images_seen)

    (TUTORIAL_DIR / "checkpoints").mkdir(exist_ok=True)
    torch.save({
        "model": model.state_dict(), "config": config,
        "loss_history": loss_history, "epoch_losses": epoch_losses,
        "example_images": example_images,
        "training_notes": f"Trained from scratch for {TRAIN_EPOCHS} cosine epochs.",
    }, TUTORIAL_DIR / "checkpoints" / "training.pt")
else:
    loss_history = checkpoint["loss_history"]
    epoch_losses = checkpoint["epoch_losses"]
    print("Demo mode: training skipped; showing the saved cosine training history.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(loss_history, alpha=0.5)
axes[0].set(xlabel="Training update", ylabel="Noise MSE", title="Batch loss")
axes[1].plot(range(1, len(epoch_losses) + 1), epoch_losses, marker="o")
axes[1].set(xlabel="Epoch", ylabel="Noise MSE", title="Mean loss per epoch")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### What has the model learned at different noise levels?

Loss mixes easy and difficult timesteps, so inspect images as well. The last row
below uses the model's noise prediction to solve for the clean image:

$$\hat x_{\mathrm{clean}}=
\frac{x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}.$$

We clip this estimate to the training data range $[-1,1]$. At high noise levels,
recovering the particular original digit is ambiguous. This single estimate is
also different from generation, which repeatedly takes small reverse steps.

**Exercise:** Complete TODO 8 to visualize the clean-image estimates.

In [ ]:
probe_steps = torch.tensor(
    [int(fraction * (diffusion.timesteps - 1)) for fraction in [0, 0.1, 0.3, 0.6, 1]],
    device=device,
)
clean = example_images[:1].to(device).repeat(len(probe_steps), 1, 1, 1)
noisy = diffusion.add_noise(clean, probe_steps, torch.randn_like(clean))
model.eval()
with torch.no_grad():
    estimates = diffusion.predict_clean(noisy, probe_steps, model(noisy, probe_steps))
fig, axes = plt.subplots(3, len(probe_steps), figsize=(10, 6))
for row, (batch, label) in enumerate(zip([clean, noisy, estimates], ["Original", "Noisy", "Estimated clean"])):
    for column, image in enumerate(batch.detach().cpu()):
        axes[row, column].imshow(image[0], cmap="gray", vmin=-1, vmax=1)
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])
        if row == 0:
            axes[row, column].set_title(f"t = {probe_steps[column].item()}")
    axes[row, 0].set_ylabel(label)
plt.tight_layout()
plt.show()

## 5. Generate digits with DDPM

Start from Gaussian noise. At each index $t$, predict noise, estimate the clean image,
and sample the preceding state. `Diffusion.denoise_step` uses:

$$\mu_t=c_{\mathrm{clean}}\hat x_{\mathrm{clean}}+c_{\mathrm{noisy}}x_t,$$
$$c_{\mathrm{clean}}=\frac{\beta_t\sqrt{\bar\alpha_{t-1}}}{1-\bar\alpha_t},
\qquad c_{\mathrm{noisy}}=\frac{\sqrt{\alpha_t}(1-\bar\alpha_{t-1})}{1-\bar\alpha_t},
\qquad \tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}.$$

For $t>0$, return $\mu_t+\sqrt{\tilde\beta_t}\,z$ with fresh Gaussian $z$.
At index `0`, return the clean estimate without extra noise. These coefficients
come from the Gaussian posterior when the clean image is known; we substitute
our predicted clean image. Sampling requires one model call per timestep.

**Exercise:** Complete TODO 9 in the diffusion module and TODO 10 in the loop below.

In [ ]:
@torch.no_grad()
def sample_ddpm(model, diffusion, initial_noise, record_every=100):
    """Generate a batch; store a few CPU snapshots for visualization."""
    was_training = model.training
    model.eval()
    x = initial_noise.clone()
    snapshots = [x[:4].cpu().clone()]
    captions = [f"Start: t={diffusion.timesteps - 1}"]
    try:
        for step in tqdm(range(diffusion.timesteps - 1, -1, -1), desc="DDPM"):
            # TODO 10: Advance the batch by one reverse diffusion step.
            # Build a (B,) long tensor filled with the current step on x.device.
            # Predict the noise with model(x, t), then call diffusion.denoise_step.
            # Assign the resulting previous image to x.
            raise NotImplementedError("TODO 10: Advance the batch by one reverse diffusion step.")
            if step % record_every == 0:
                snapshots.append(x[:4].cpu().clone())
                captions.append("Clean" if step == 0 else f"t={step - 1}")
    finally:
        model.train(was_training)
    return x, snapshots, captions


# Fixed starting noise and reverse-process seed make checkpoint comparisons repeatable.
initial_noise = torch.randn(
    NUM_SAMPLES, 1, 28, 28, generator=torch.Generator().manual_seed(123)
).to(device)
torch.manual_seed(1234)
samples, snapshots, captions = sample_ddpm(model, diffusion, initial_noise)
show_images(samples, "Generated digits — DDPM, cosine schedule")

In [ ]:
# One sample per row, time moving left to right. Include the initial noise.
fig, axes = plt.subplots(len(snapshots[0]), len(snapshots), figsize=(16, 5), squeeze=False)
for column, (batch, caption) in enumerate(zip(snapshots, captions)):
    for row in range(len(batch)):
        axes[row, column].imshow(batch[row, 0], cmap="gray", vmin=-1, vmax=1)
        axes[row, column].axis("off")
        if row == 0:
            axes[row, column].set_title(caption, fontsize=9)
plt.suptitle("Reverse diffusion: noise → digits")
plt.tight_layout()
plt.show()

## 6. Optional: faster sampling with DDIM

DDIM uses the same model at a smaller set of timesteps. It adds no fresh random
noise after initialization. Fewer model calls make it faster, but quality can change.
Simply skipping DDPM steps would not give the same update.

$$x_s=\sqrt{\bar\alpha_s}\,\hat x_{\mathrm{clean}}
     +\sqrt{1-\bar\alpha_s}\,\varepsilon_\theta(x_t,t).$$

Because we clip the clean estimate to $[-1,1]$, we recompute its corresponding
noise as $(x_t-\sqrt{\bar\alpha_t}\,\hat x_{\mathrm{clean}})/\sqrt{1-\bar\alpha_t}$
before this update. This keeps the two estimates consistent with the current image.
At the end, return the clean estimate. This deterministic variant adds no new
random noise after initialization. Fewer model calls make sampling faster, but
quality can change; experiment with 20, 50, and 100 steps. DDPM still uses every
step; simply skipping indices in its loop is not the DDIM update.

In [ ]:
@torch.no_grad()
def sample_ddim(model, diffusion, initial_noise, steps=50):
    """Deterministic DDIM using 2 <= steps <= diffusion.timesteps."""
    was_training = model.training
    model.eval()
    x = initial_noise.clone()
    times = torch.linspace(diffusion.timesteps - 1, 0, steps).long().tolist()
    try:
        for index, step in enumerate(tqdm(times, desc="DDIM")):
            t = torch.full((len(x),), step, device=x.device, dtype=torch.long)
            predicted_noise = model(x, t)
            clean = diffusion.predict_clean(x, t, predicted_noise)
            if index == len(times) - 1:
                x = clean
            else:
                # Clipping changed the clean estimate; keep its noise estimate consistent.
                alpha_bar = diffusion.alpha_bar[step]
                predicted_noise = (x - alpha_bar.sqrt() * clean) / (1 - alpha_bar).sqrt()
                alpha_bar_previous = diffusion.alpha_bar[times[index + 1]]
                x = (alpha_bar_previous.sqrt() * clean
                     + (1 - alpha_bar_previous).sqrt() * predicted_noise)
    finally:
        model.train(was_training)
    return x

In [ ]:
fast_samples = sample_ddim(model, diffusion, initial_noise, steps=50)
show_images(fast_samples, "DDIM: 50 steps, same starting noise as DDPM")

## Discuss together

- Why does training use one random timestep while DDPM generation uses a sequence?
- Why is the training target noise rather than the digit label?
- Why is the original digit ambiguous at high noise levels?
- Which image details do the two U-Net skip connections preserve?
- Compare DDPM and DDIM: what changes in speed and digit shapes?

Loss and sample grids are useful observations, not proof that a model generalizes.
When finished, compare your implementation with the completed reference.
Which part was most sensitive to tensor shapes? What changes when you vary the noise level?